# VLM-SAM3 Results — Jobs 465388 & 465389

**Job 465388** — EXP-C-box / EXP-C-box-cot (VLM → bbox → SAM3 geometry prompt)  
**Job 465389** — EXP-F-mf1 / mf3 / mf5 / mf10 (multiframe source ablation)  
**Completed**: 2026-03-09  

---

## Task Definition

Given a **source frame** (ego or exo camera) with a **known object annotation**, find and segment the **same object** in a **destination frame** from a *different viewpoint* (ego↔exo cross-view correspondence).

```
Source frame  →  VLM  →  object description or bbox
                                  ↓
Destination frame  →  SAM3  →  predicted mask
                                  ↓
                       IoU vs ground truth mask
```

**Baseline**: LM-EEC (Language-guided Mask Estimation with Ego-Exo Correspondence) — a traditional non-VLM method that achieves mean IoU = **0.660** (exo→ego direction).

In [ ]:
from pathlib import Path

REPO_ROOT  = Path("/home/3164542/reluminati-research")
SCRIPT_DIR = REPO_ROOT / "src/scripts/experiments/vlm-sam3"
DATA_ROOT  = Path("/data/video_datasets/3321908/output_dir_all")

RESULT_FILES = [
    SCRIPT_DIR / "results_463119.csv",
    SCRIPT_DIR / "results_465254.csv",
    SCRIPT_DIR / "results_465301.csv",
    SCRIPT_DIR / "results_465302.csv",
    SCRIPT_DIR / "results_465303.csv",
    SCRIPT_DIR / "results_465388.csv",
    SCRIPT_DIR / "results_465389.csv",
]

BOX_EXPS       = ["EXP-C-box", "EXP-C-box-cot"]
MF_EXPS        = ["EXP-F-mf1", "EXP-F-mf3", "EXP-F-mf5", "EXP-F-mf10"]
TEXT_BASELINES = ["EXP-C", "EXP-C-cot"]
ALL_EXPS       = TEXT_BASELINES + BOX_EXPS + MF_EXPS

LM_EEC = 0.660
KEY_COLS = ["take_uid","object_name","src_camera","dest_camera","frame","experiment"]

print("Config OK.")

In [ ]:
import json, sys, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.cm as cm
import seaborn as sns
from PIL import Image

# Add experiment.py to path so we can reuse its mask utilities
sys.path.insert(0, str(SCRIPT_DIR))
from experiment import load_annotation, get_mask, decode_rle, _align_mask

warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.figsize": (12, 5), "font.size": 11})
sns.set_theme(style="whitegrid", palette="husl")

EXP_PALETTE = {
    "EXP-C":         "#3498db",
    "EXP-C-cot":     "#e74c3c",
    "EXP-C-box":     "#f39c12",
    "EXP-C-box-cot": "#e67e22",
    "EXP-F-mf1":     "#27ae60",
    "EXP-F-mf3":     "#16a085",
    "EXP-F-mf5":     "#2980b9",
    "EXP-F-mf10":    "#8e44ad",
}

EXP_LABELS = {
    "EXP-C":         "EXP-C\n(cloud,text)",
    "EXP-C-cot":     "EXP-C-cot\n(instruct,text)",
    "EXP-C-box":     "EXP-C-box\n(cloud,bbox)",
    "EXP-C-box-cot": "EXP-C-box-cot\n(instruct,bbox)",
    "EXP-F-mf1":     "mf1 (1 frame)",
    "EXP-F-mf3":     "mf3 (3 frames)",
    "EXP-F-mf5":     "mf5 (5 frames)",
    "EXP-F-mf10":    "mf10 (10 frames)",
}

print("Imports OK.")

---
## 1  Load & Deduplicate

In [ ]:
dfs = []
for f in RESULT_FILES:
    if f.exists():
        tmp = pd.read_csv(f, dtype=str)
        dfs.append(tmp)
        print(f"  {f.name:45s}  {len(tmp):6d} rows")
    else:
        print(f"  SKIP (missing): {f.name}")

raw = pd.concat(dfs, ignore_index=True)
all_df = raw.drop_duplicates(subset=KEY_COLS, keep="last").copy()
all_df["iou"] = pd.to_numeric(all_df["iou"], errors="coerce")
all_df["success"] = all_df["iou"] > 0

print(f"\nTotal unique (pair × experiment): {len(all_df):,}")
print(f"Experiments: {sorted(all_df['experiment'].unique())}")

---
## 2  Experiment Design — What Each Experiment Does

All experiments share the same **task** (cross-view object segmentation) and **SAM3 model**, but differ in how the VLM is used and what it produces.

### 2.1 Shared Architecture

```
SOURCE FRAME(S)          DESTINATION FRAME
     │                          │
     ▼                          │
  Qwen3-VL 235B  ─────────────►│
  (sees: src annotated          │
   + src clean                  │
   + dst clean)                 │
     │                          │
     ▼ text / bbox              ▼
     └──────────────►  SAM3  ──►  predicted mask
                                         │
                                         ▼
                               IoU vs ground truth
```

**SAM3** (Segment Anything Model 3) can accept two types of prompts:
- **Text prompt**: fed through a CLIP ViT-L/14 text encoder (hard limit: ~28 usable tokens)
- **Box prompt**: fed through a *geometry encoder* (ROI-align + sinusoidal position encoding + cross-attention) — no token limit

---

### 2.2 EXP-C — Cloud model, plain text prompt
```
VLM model    : Qwen3-VL 235b-cloud  (faster, num_predict=64)
VLM input    : src_annotated + src_clean + dst_clean  (3 images)
VLM output   : short text description of object in dst ("red basketball")
SAM3 prompt  : text
```
The VLM sees the annotated source frame (object highlighted with a colored overlay), the clean source, and the clean destination. It must describe the object *as it appears in the destination frame*. SAM3 then uses this description to find and segment the object.

**Limitation**: SAM3's text encoder only handles ~28 tokens. Short descriptions like "basketball" work; longer ones get truncated.

---

### 2.3 EXP-C-cot — Instruct model, chain-of-thought text prompt
```
VLM model    : Qwen3-VL 235b-instruct  (reasoning model, num_predict=512)
VLM input    : same 3 images
VLM output   : CoT reasoning + final answer ("final answer: blue basketball")
SAM3 prompt  : only the final answer (extracted from CoT, truncated to 28 tokens)
```
The instruct model reasons step by step before giving a final answer. We extract only the final answer and feed it to SAM3. Zero missing outputs (512 token budget is plenty).

---

### 2.4 EXP-C-box — Cloud model, bbox output → SAM3 geometry prompt  ⭐ NEW
```
VLM model    : Qwen3-VL 235b-cloud  (num_predict=64)
VLM input    : same 3 images
VLM output   : bounding box coordinates  "x1,y1,x2,y2"  (in 0-1000 space)
SAM3 prompt  : input_boxes=[[[x1,y1,x2,y2]]]  (geometry encoder, no text)
```
Instead of describing the object, the VLM *localizes* it: outputs pixel coordinates of the bounding box around the object in the destination frame. SAM3 uses its geometry encoder to segment from the box — **bypassing the 28-token limit entirely**.

**Key fix**: Qwen3-VL always outputs coordinates in 0-1000 normalized space (not pixels). Must divide by 1000 and multiply by image dimensions before passing to SAM3.

---

### 2.5 EXP-C-box-cot — Instruct model, CoT + bbox output
```
VLM model    : Qwen3-VL 235b-instruct  (num_predict=512)
VLM output   : CoT reasoning → final bbox coordinates
SAM3 prompt  : extracted bbox from CoT output
```
Same as EXP-C-box but with chain-of-thought reasoning. Hypothesis: reasoning helps localize better. **Result**: it actually hurts (30.4% vs 49.4%) — the model second-guesses its coordinate estimates.

---

### 2.6 EXP-F-mf{1,3,5,10} — Multiframe ablation  ⭐ NEW
```
VLM model    : Qwen3-VL 235b-instruct
VLM input    : N source frames (N=1,3,5,10) + dst_clean
VLM output   : text description
SAM3 prompt  : text
```
Instead of 1 source frame, we show N preceding annotated frames. Hypothesis: temporal context helps the VLM understand what the object looks like across viewpoints. The N frames are sampled uniformly from the annotated frames *before* the current frame (temporal consistency preserved).

---
## 3  Overall Results Summary

In [ ]:
def exp_stats(df, exp):
    g = df[df["experiment"] == exp].copy()
    iou = g["iou"].dropna()
    pos = iou[iou > 0]
    n = len(iou)
    return {
        "N":           n,
        "iou>0 %":     round(100 * len(pos) / max(1, n), 1),
        "mean_all":    round(iou.mean(), 3),
        "mean_pos":    round(pos.mean(), 3) if len(pos) else float("nan"),
        "iou>0.5 %":  round(100 * (pos > 0.5).mean(), 1) if len(pos) else 0.0,
        "iou>0.75 %": round(100 * (pos > 0.75).mean(), 1) if len(pos) else 0.0,
    }

rows = []
for exp in ALL_EXPS:
    s = exp_stats(all_df, exp)
    s["experiment"] = exp
    rows.append(s)
summ = pd.DataFrame(rows).set_index("experiment")

print(f"{'Experiment':<20} {'N':>5}  {'iou>0%':>7}  {'mean_all':>9}  {'mean_pos':>9}  {'iou>0.5%':>9}  {'iou>0.75%':>10}")
print("-"*80)
print(f"{'LM-EEC (baseline)':<20} {'—':>5}  {'—':>7}  {'0.660':>9}  {'0.660':>9}  {'—':>9}  {'—':>10}")
for exp in ALL_EXPS:
    s = summ.loc[exp]
    mp = f"{s['mean_pos']:.3f}" if not np.isnan(s['mean_pos']) else "nan"
    partial = " *" if s['N'] < 1900 else ""
    print(f"  {exp:<18} {s['N']:>5}  {s['iou>0 %']:>6.1f}%  {s['mean_all']:>9.3f}  {mp:>9}  {s['iou>0.5 %']:>8.1f}%  {s['iou>0.75 %']:>9.1f}%{partial}")
print("\n* = partial run (~75% of 1910 pairs)")

In [ ]:
# Styled table — numeric only (no mixed types for gradient)
styled = summ.copy()
display(
    styled.style
    .format({
        "N": "{:,}",
        "iou>0 %":   "{:.1f}%",
        "mean_all":  "{:.3f}",
        "mean_pos":  "{:.3f}",
        "iou>0.5 %": "{:.1f}%",
        "iou>0.75 %":"{:.1f}%",
    }, na_rep="—")
    .background_gradient(subset=["iou>0 %"],   cmap="RdYlGn", vmin=0, vmax=60)
    .background_gradient(subset=["mean_pos"],   cmap="YlGn",  vmin=0.2, vmax=0.75)
    .background_gradient(subset=["iou>0.5 %"], cmap="YlGn",  vmin=0, vmax=80)
    .set_caption("All experiments — deduplicated across jobs 463119, 465254, 465301-303, 465388, 465389")
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

labels = [EXP_LABELS[e] for e in ALL_EXPS]
colors = [EXP_PALETTE[e] for e in ALL_EXPS]
x = np.arange(len(ALL_EXPS))

# Localization rate
ax = axes[0]
vals = [summ.loc[e, "iou>0 %"] for e in ALL_EXPS]
bars = ax.bar(x, vals, color=colors, edgecolor="white", width=0.7)
for b, v in zip(bars, vals):
    ax.text(b.get_x()+b.get_width()/2, v+0.5, f"{v:.0f}%",
            ha="center", va="bottom", fontsize=8, fontweight="bold")
ax.set_xticks(x); ax.set_xticklabels(labels, fontsize=8, rotation=30, ha="right")
ax.set_ylabel("IoU > 0 (%)")
ax.set_title("Localization success rate", fontweight="bold")

# Quality (mean IoU | positive)
ax = axes[1]
vals2 = [summ.loc[e, "mean_pos"] for e in ALL_EXPS]
bars2 = ax.bar(x, vals2, color=colors, edgecolor="white", width=0.7)
ax.axhline(LM_EEC, ls="--", color="black", lw=2, label=f"LM-EEC ({LM_EEC})")
for b, v in zip(bars2, vals2):
    if not np.isnan(v):
        ax.text(b.get_x()+b.get_width()/2, v+0.005, f"{v:.3f}",
                ha="center", va="bottom", fontsize=8, fontweight="bold")
ax.set_xticks(x); ax.set_xticklabels(labels, fontsize=8, rotation=30, ha="right")
ax.set_ylabel("Mean IoU (positive pairs only)")
ax.set_title("Segmentation quality when successful", fontweight="bold")
ax.legend()

plt.suptitle("VLM-SAM3 — Localization vs Quality trade-off", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

print("\nKey insight:")
print(f"  EXP-C-box : highest localization ({summ.loc['EXP-C-box','iou>0 %']:.1f}%) but lower quality ({summ.loc['EXP-C-box','mean_pos']:.3f})")
print(f"  EXP-C     : lower localization ({summ.loc['EXP-C','iou>0 %']:.1f}%) but beats LM-EEC ({summ.loc['EXP-C','mean_pos']:.3f} > {LM_EEC})")
print(f"  → Bottleneck is VLM localization, not SAM3 segmentation")

---
## 4  EXP-C-box Deep Dive

Three-way outcome breakdown for each pair:
1. **VLM says not visible** → SAM3 not called → IoU = 0
2. **VLM outputs bbox, but SAM3 mask doesn't overlap GT** → IoU = 0  
3. **Success**: IoU > 0

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

for col, exp in enumerate(BOX_EXPS):
    g = all_df[all_df["experiment"] == exp].copy()
    iou = g["iou"].dropna()
    pos = iou[iou > 0]
    has_bbox = g["vlm_bbox"].notna() & (g["vlm_bbox"].astype(str) != "nan")
    not_vis  = (~has_bbox).sum()
    iou_zero = (iou == 0).sum()

    # Stacked bar: outcome breakdown
    ax = axes[0, col]
    cats   = ["Not visible (VLM)", "bbox found, IoU=0", "IoU > 0 (success)"]
    vals   = [not_vis, iou_zero, len(pos)]
    colors = ["#e74c3c", "#f39c12", "#2ecc71"]
    bottom = 0
    for cat, v, c in zip(cats, vals, colors):
        bar = ax.bar([exp], [v], bottom=bottom, color=c, edgecolor="white", label=cat)
        if v > 20:
            ax.text(0, bottom + v/2, f"{v}\n({100*v/len(g):.0f}%)",
                    ha="center", va="center", fontsize=10, fontweight="bold", color="white")
        bottom += v
    ax.set_ylabel("# pairs")
    ax.set_title(f"{exp} — outcome breakdown (N={len(g)})", fontweight="bold")
    ax.legend(loc="upper right", fontsize=8)

    # IoU distribution (positive only)
    ax = axes[1, col]
    buckets = {
        "(0, 0.1]": ((pos>0) & (pos<=0.1)).sum(),
        "(0.1, 0.25]": ((pos>0.1) & (pos<=0.25)).sum(),
        "(0.25, 0.5]": ((pos>0.25) & (pos<=0.5)).sum(),
        "(0.5, 0.75]": ((pos>0.5) & (pos<=0.75)).sum(),
        "(0.75, 1.0]": (pos>0.75).sum(),
    }
    bcolors = ["#e74c3c","#e67e22","#f1c40f","#2ecc71","#27ae60"]
    bars = ax.bar(buckets.keys(), buckets.values(), color=bcolors, edgecolor="white")
    for b, v in zip(bars, buckets.values()):
        ax.text(b.get_x()+b.get_width()/2, v+1, str(v),
                ha="center", fontsize=9, fontweight="bold")
    ax.set_xlabel("IoU bucket")
    ax.set_ylabel("# pairs")
    ax.set_title(f"{exp} — IoU distribution (positive, n={len(pos)})", fontweight="bold")
    ax.tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.show()

for exp in BOX_EXPS:
    g = all_df[all_df["experiment"] == exp]
    iou = g["iou"].dropna()
    pos = iou[iou > 0]
    has_bbox = g["vlm_bbox"].notna() & (g["vlm_bbox"].astype(str) != "nan")
    print(f"\n{exp}:")
    print(f"  VLM found bbox  : {has_bbox.sum()} ({100*has_bbox.mean():.1f}%)")
    print(f"  iou>0           : {len(pos)}/{len(iou)} = {100*len(pos)/max(1,len(iou)):.1f}%")
    print(f"  mean IoU (pos)  : {pos.mean():.3f}   median: {pos.median():.3f}")
    print(f"  iou>0.5         : {(pos>0.5).sum()} ({100*(pos>0.5).mean():.1f}%)")
    print(f"  iou>0.75        : {(pos>0.75).sum()} ({100*(pos>0.75).mean():.1f}%)")
    print(f"  iou>0.90        : {(pos>0.9).sum()} ({100*(pos>0.9).mean():.1f}%)")

---
## 5  Bbox vs Text: the Localization–Quality Trade-off

In [ ]:
compare_exps = ["EXP-C", "EXP-C-cot", "EXP-C-box", "EXP-C-box-cot"]
labels4 = [EXP_LABELS[e] for e in compare_exps]
colors4 = [EXP_PALETTE[e] for e in compare_exps]

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
metrics = [
    ("iou>0 %",   "Localization rate (IoU>0 %)",    "Localization success"),
    ("mean_pos",  "Mean IoU | positive pairs",        "Segmentation quality"),
    ("iou>0.5 %","% pairs with IoU > 0.5",           "High-quality mask rate"),
]
for ax, (col, ylabel, title) in zip(axes, metrics):
    vals = [summ.loc[e, col] for e in compare_exps]
    bars = ax.bar(range(4), vals, color=colors4, edgecolor="white", width=0.6)
    for i, (b, v) in enumerate(zip(bars, vals)):
        if not np.isnan(float(v)):
            ax.text(i, float(v)+0.4, f"{float(v):.1f}",
                    ha="center", fontsize=10, fontweight="bold")
    if col == "mean_pos":
        ax.axhline(LM_EEC, ls="--", color="black", lw=2, label=f"LM-EEC ({LM_EEC})")
        ax.legend(fontsize=9)
    ax.set_xticks(range(4))
    ax.set_xticklabels(labels4, fontsize=9, rotation=15, ha="right")
    ax.set_ylabel(ylabel, fontsize=9)
    ax.set_title(title, fontweight="bold")

handles = [mpatches.Patch(color=EXP_PALETTE[e], label=e) for e in compare_exps]
fig.legend(handles=handles, loc="upper center", ncol=4, bbox_to_anchor=(0.5, 1.03))
plt.suptitle("Bbox vs Text — Localization wins vs Quality wins",
             fontsize=13, fontweight="bold", y=1.06)
plt.tight_layout()
plt.show()

---
## 6  Per-Scenario Breakdown (EXP-C-box)

In [ ]:
SCENARIOS = {
    "basketball": ["basketball","hoop","Basket"],
    "cooking":    ["knife","pan","bowl","kettle","plate","board","noodle",
                   "egg","paper","fork","spoon","mug","strainer","spatula","spaghetti"],
    "health":     ["CPR","dummy","covid","swab","test cassette","test device",
                   "antigen","extraction","solution tube","test kit","instruction manual"],
    "music":      ["piano","sheet","guitar"],
    "bike":       ["bike","bicycle","wrench","socket","pump","tube",
                   "handlebar","grip","stand","tire","trolley"],
}

box_df = all_df[all_df["experiment"]=="EXP-C-box"].copy()

scen_rows = []
for sc, kws in SCENARIOS.items():
    mask = box_df["object_name"].str.contains("|".join(kws), case=False, na=False)
    g    = box_df[mask]
    iou  = g["iou"].dropna()
    pos  = iou[iou > 0]
    scen_rows.append({
        "scenario":    sc,
        "n":           len(g),
        "iou>0 %":     100*len(pos)/max(1,len(iou)),
        "mean_pos":    pos.mean() if len(pos) else float("nan"),
        "iou>0.5 %":  100*(pos>0.5).mean() if len(pos) else 0.0,
    })
scen_df = pd.DataFrame(scen_rows).set_index("scenario")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
SCEN_COLORS = {"basketball":"#e74c3c","cooking":"#f39c12",
               "health":"#3498db","music":"#9b59b6","bike":"#2ecc71"}
scols = [SCEN_COLORS[s] for s in scen_df.index]

for ax, col, title in [
    (axes[0], "iou>0 %",  "Localization success rate (%)"),
    (axes[1], "mean_pos", "Mean IoU | positive"),
]:
    bars = ax.bar(scen_df.index, scen_df[col], color=scols, edgecolor="white")
    for b, v in zip(bars, scen_df[col]):
        if not np.isnan(float(v)):
            ax.text(b.get_x()+b.get_width()/2, float(v)+0.5,
                    f"{float(v):.1f}", ha="center", fontsize=10, fontweight="bold")
    if col == "mean_pos":
        ax.axhline(LM_EEC, ls="--", color="black", lw=2, label=f"LM-EEC ({LM_EEC})")
        ax.legend()
    ax.set_title(f"EXP-C-box — {title}", fontweight="bold")

plt.tight_layout()
plt.show()

display(scen_df.style
    .format({"iou>0 %":"{:.1f}%","mean_pos":"{:.3f}","iou>0.5 %":"{:.1f}%"}, na_rep="—")
    .background_gradient(subset=["iou>0 %"], cmap="RdYlGn")
)

---
## 7  Multiframe Ablation (EXP-F)

**Hypothesis**: Showing more annotated source frames helps the VLM understand the object's appearance across time, improving localization.  
**Result**: The opposite — fewer frames = better localization.

In [ ]:
mf_rows = []
for exp in MF_EXPS:
    g   = all_df[all_df["experiment"]==exp]
    iou = g["iou"].dropna()
    pos = iou[iou > 0]
    mf_rows.append({
        "exp":        exp,
        "n_frames":   int(exp.replace("EXP-F-mf","")),
        "N":          len(iou),
        "iou>0 %":    100*len(pos)/max(1,len(iou)),
        "mean_pos":   pos.mean() if len(pos) else float("nan"),
        "iou>0.5 %": 100*(pos>0.5).mean() if len(pos) else 0.0,
        "iou>0.75 %":100*(pos>0.75).mean() if len(pos) else 0.0,
    })
mf_df = pd.DataFrame(mf_rows)

display(mf_df.set_index("exp").style
    .format({"iou>0 %":"{:.1f}%","mean_pos":"{:.3f}",
             "iou>0.5 %":"{:.1f}%","iou>0.75 %":"{:.1f}%"}, na_rep="—")
    .background_gradient(subset=["iou>0 %"],  cmap="RdYlGn")
    .background_gradient(subset=["mean_pos"],  cmap="YlGn", vmin=0.4, vmax=0.7)
)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
x = mf_df["n_frames"]

ax = axes[0]
ax.plot(x, mf_df["iou>0 %"], "o-", color="#e74c3c", lw=2.5, ms=10, zorder=3)
for xi, yi in zip(x, mf_df["iou>0 %"]):
    ax.annotate(f"{yi:.1f}%", (xi,yi), xytext=(0,10), textcoords="offset points",
                ha="center", fontsize=11, fontweight="bold")
ax.set_xlabel("# Source Frames"); ax.set_ylabel("IoU > 0 (%)")
ax.set_xticks([1,3,5,10]); ax.set_title("Localization rate vs # frames", fontweight="bold")
ax.annotate("More frames\n= VLM confused?", xy=(5,17.6), xytext=(6.5,22),
            arrowprops=dict(arrowstyle="->",color="gray"), fontsize=9, color="gray")

ax = axes[1]
ax.plot(x, mf_df["mean_pos"], "s-", color="#27ae60", lw=2.5, ms=10, label="mean IoU | success")
ax.axhline(LM_EEC, ls="--", color="black", lw=2, label=f"LM-EEC ({LM_EEC})")
for xi, yi in zip(x, mf_df["mean_pos"]):
    ax.annotate(f"{yi:.3f}", (xi,yi), xytext=(0,-18), textcoords="offset points",
                ha="center", fontsize=10, fontweight="bold")
ax.set_xlabel("# Source Frames"); ax.set_ylabel("Mean IoU | positive")
ax.set_xticks([1,3,5,10]); ax.set_title("Quality vs # frames", fontweight="bold")
ax.legend()
ax.annotate("3 frames = sweet spot\nfor quality", xy=(3,0.620), xytext=(5,0.58),
            arrowprops=dict(arrowstyle="->",color="gray"), fontsize=9, color="gray")

plt.suptitle("Multiframe ablation — EXP-F-mf{1,3,5,10}", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

# Violin of IoU distributions
mf_pos = all_df[all_df["experiment"].isin(MF_EXPS) & (all_df["iou"]>0)]
fig, ax = plt.subplots(figsize=(10, 4))
sns.violinplot(data=mf_pos, x="experiment", y="iou", order=MF_EXPS,
               palette={e:EXP_PALETTE[e] for e in MF_EXPS}, inner="quartile", cut=0, ax=ax)
ax.axhline(LM_EEC, ls="--", color="black", lw=2, label=f"LM-EEC ({LM_EEC})")
ax.set_xticklabels([f"{e}\n(n={len(mf_pos[mf_pos['experiment']==e])})"
                    for e in MF_EXPS])
ax.set_title("IoU distribution — positive pairs only", fontweight="bold")
ax.legend()
plt.tight_layout()
plt.show()

---
## 8  Visual Examples — Frame-by-Frame Inspection

For each selected pair we show:
- **Source frame** (with ground-truth mask overlay in red)
- **Destination frame** (clean)
- **Destination frame** (ground truth mask in green + predicted mask in blue)
- **VLM bbox** (orange rectangle, EXP-C-box only)
- **IoU score**

In [ ]:
import cv2

def find_image(take_uid, camera, frame):
    """Resolve image path."""
    for ext in [".jpg", ".png", ""]:
        p = DATA_ROOT / take_uid / camera / f"{frame}{ext}"
        if p.exists():
            return p
    return None

def overlay_mask(img_np, mask, color, alpha=0.45):
    """Blend a boolean mask over an RGB image. Resizes mask to match image if needed."""
    H, W = img_np.shape[:2]
    if mask.shape[0] != H or mask.shape[1] != W:
        mask = cv2.resize(mask.astype(np.uint8), (W, H),
                          interpolation=cv2.INTER_NEAREST).astype(bool)
    out = img_np.copy().astype(float)
    for c, col_val in enumerate(color):
        out[:, :, c] = np.where(mask,
                                 out[:, :, c] * (1 - alpha) + col_val * alpha,
                                 out[:, :, c])
    return out.clip(0, 255).astype(np.uint8)

def draw_box(img_np, box_str, color=(255, 140, 0), lw=3):
    """Draw bounding box from 'x1,y1,x2,y2' string (already in pixel coords)."""
    if not box_str or str(box_str) in ("nan", "None", ""):
        return img_np
    try:
        x1, y1, x2, y2 = [int(float(v)) for v in str(box_str).split(",")]
        img = img_np.copy()
        cv2.rectangle(img, (x1, y1), (x2, y2), color, lw)
        return img
    except Exception:
        return img_np

def show_pair(row, exp_name, ann_cache={}):
    """Show one result: source frame | dest clean | dest GT+bbox."""
    uid     = row["take_uid"]
    obj     = row["object_name"]
    src_cam = row["src_camera"]
    dst_cam = row["dest_camera"]
    frame   = str(row["frame"])
    iou_val = float(row["iou"]) if not pd.isna(row.get("iou", float("nan"))) else 0.0
    vlm_out = str(row.get("vlm_output", ""))[:60]
    vlm_bbox = str(row.get("vlm_bbox", "nan"))

    if uid not in ann_cache:
        try:
            ann_cache[uid] = load_annotation(str(DATA_ROOT), uid)
        except Exception:
            ann_cache[uid] = None
    ann = ann_cache[uid]

    src_path = find_image(uid, src_cam, frame)
    dst_path = find_image(uid, dst_cam, frame)
    if src_path is None or dst_path is None:
        print(f"  Image not found: {uid} / cam={src_cam}/{dst_cam} / frame={frame}")
        return

    src_img = np.array(Image.open(src_path).convert("RGB"))
    dst_img = np.array(Image.open(dst_path).convert("RGB"))

    src_mask = get_mask(ann, obj, src_cam, frame) if ann else None
    dst_gt   = get_mask(ann, obj, dst_cam, frame) if ann else None

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    # 1 — source + GT mask (red)
    src_vis = overlay_mask(src_img, src_mask, [255, 50, 50]) if src_mask is not None else src_img
    axes[0].imshow(src_vis)
    axes[0].set_title(f"Source ({src_cam})\nGT mask in red", fontsize=9)
    axes[0].axis("off")

    # 2 — destination clean
    axes[1].imshow(dst_img)
    axes[1].set_title(f"Destination ({dst_cam})\nclean", fontsize=9)
    axes[1].axis("off")

    # 3 — destination + GT mask (green) + VLM bbox (orange)
    dst_vis = overlay_mask(dst_img, dst_gt, [50, 200, 50]) if dst_gt is not None else dst_img.copy()
    if vlm_bbox not in ("nan", "None", ""):
        dst_vis = draw_box(dst_vis, vlm_bbox, color=(255, 140, 0), lw=3)
    axes[2].imshow(dst_vis)
    bbox_note = "\nVLM bbox (orange)" if vlm_bbox not in ("nan", "None", "") else ""
    axes[2].set_title(f"Destination\nGT (green){bbox_note}", fontsize=9)
    axes[2].axis("off")

    iou_color = "#2ecc71" if iou_val >= 0.5 else ("#f39c12" if iou_val > 0 else "#e74c3c")
    fig.suptitle(
        f"{exp_name}  |  {obj}  |  frame {frame}\n"
        f"VLM: {vlm_out}\n"
        f"IoU = {iou_val:.3f}",
        fontsize=9, fontweight="bold", color=iou_color if iou_val > 0 else "black"
    )
    plt.tight_layout()
    plt.show()

print("Visualization utilities loaded.")

In [ ]:
# ── Top-5 EXP-C-box by IoU ──────────────────────────────────────────────────
print("="*60)
print("TOP 5 EXP-C-box results (highest IoU)")
print("="*60)

top5 = all_df[all_df["experiment"]=="EXP-C-box"].nlargest(5, "iou")
for _, row in top5.iterrows():
    print(f"  [{row['object_name']}]  frame={row['frame']}  "
          f"vlm_bbox={row['vlm_bbox']}  IoU={float(row['iou']):.3f}")
    show_pair(row, "EXP-C-box")

In [ ]:
# ── Failures: bbox found but IoU=0 ──────────────────────────────────────────
print("="*60)
print("EXP-C-box FAILURES: bbox found but IoU = 0")
print("(VLM localized but SAM3 mask didn't overlap GT)")
print("="*60)

box_df_full = all_df[all_df["experiment"]=="EXP-C-box"].copy()
fails = box_df_full[
    (box_df_full["vlm_bbox"].notna()) &
    (box_df_full["vlm_bbox"].astype(str) != "nan") &
    (box_df_full["iou"] == 0)
].head(3)

for _, row in fails.iterrows():
    print(f"  [{row['object_name']}]  frame={row['frame']}  "
          f"vlm_bbox={row['vlm_bbox']}  IoU=0")
    show_pair(row, "EXP-C-box")

In [ ]:
# ── Same pair: EXP-C-box vs EXP-C text comparison ────────────────────────────
print("="*60)
print("SAME PAIR COMPARISON: EXP-C-box vs EXP-C")
print("="*60)

pair_keys = ["take_uid", "object_name", "src_camera", "dest_camera", "frame"]

box_data  = all_df[all_df["experiment"] == "EXP-C-box"].copy()
text_data = all_df[all_df["experiment"] == "EXP-C"].copy()

# Merge on pair keys
merged = box_data.merge(
    text_data[pair_keys + ["iou", "vlm_output"]],
    on=pair_keys, suffixes=("_box", "_text")
)
merged["iou_box"]  = pd.to_numeric(merged["iou_box"],  errors="coerce").fillna(0)
merged["iou_text"] = pd.to_numeric(merged["iou_text"], errors="coerce").fillna(0)
merged["diff"]     = merged["iou_box"] - merged["iou_text"]
print(f"Shared pairs: {len(merged)}")

for label, sub in [
    ("Box wins (box IoU >> text IoU)", merged.nlargest(3, "diff")),
    ("Text wins (text IoU >> box IoU)", merged.nsmallest(3, "diff")),
]:
    print(f"\n── {label} ──")
    for _, row in sub.iterrows():
        print(f"  [{row['object_name']}]  frame={row['frame']}  "
              f"IoU_box={row['iou_box']:.3f}  IoU_text={row['iou_text']:.3f}  "
              f"text='{str(row['vlm_output_text'])[:40]}'")
        show_pair(row, "EXP-C-box")

In [ ]:
# ── Multiframe: show best mf1 and best mf3 examples ─────────────────────────
print("="*60)
print("MULTIFRAME: Best examples from mf1 and mf3")
print("="*60)

for exp in ["EXP-F-mf1", "EXP-F-mf3"]:
    print(f"\n── {exp} — top 2 ──")
    top = all_df[all_df["experiment"]==exp].nlargest(2, "iou")
    for _, row in top.iterrows():
        print(f"  [{row['object_name']}]  IoU={float(row['iou']):.3f}  "
              f"vlm='{str(row['vlm_output'])[:50]}'")
        show_pair(row, exp)

---
## 9  Key Findings & Conclusions

In [ ]:
box  = all_df[all_df["experiment"]=="EXP-C-box"]
expc = all_df[all_df["experiment"]=="EXP-C"]

def pct(df): return 100*(df["iou"]>0).mean()
def mp(df): p=df[df["iou"]>0]["iou"]; return p.mean()

print("="*70)
print("  FINAL FINDINGS — Jobs 465388 & 465389")
print("="*70)
print(f"""
  1. BEST LOCALIZATION: EXP-C-box (bbox approach)
     {pct(box):.1f}% of pairs successfully found and segmented
     → VLM outputs x,y coordinates instead of text → SAM3 geometry prompt
     → Bypasses the 28-token text encoder limit

  2. BEST QUALITY: EXP-C (text, plain)
     Mean IoU = {mp(expc):.3f} on success cases (beats LM-EEC {LM_EEC}!)
     → When VLM gives a good description, SAM3 segments precisely
     → 78.1% of successful pairs have IoU > 0.5

  3. THE TRADE-OFF:
     Bbox approach finds MORE objects ({pct(box):.1f}%) but SAM3 boxes are
     less precise than SAM3 text prompts ({mp(box):.3f} vs {mp(expc):.3f})

  4. BOTTLENECK: VLM localization (not SAM3 segmentation)
     When the VLM correctly identifies the object, SAM3 handles it well.
     The pipeline fails when the VLM outputs a wrong/empty description.

  5. COT HURTS BBOX (49.4% → 30.4%)
     Reasoning makes the model second-guess coordinate estimates.
     For bbox outputs, direct prediction > chain-of-thought.

  6. MULTIFRAME: FEWER = BETTER for localization
     mf1 (33.1%) > mf3 (23.4%) > mf5 (17.6%)
     More images = VLM gets confused by visual clutter?
     But mf3 gives best quality when found (mean IoU=0.620)

  7. PER SCENARIO (EXP-C-box):
     Music  : 96.9% (piano/guitar — large, distinct objects)
     Basketball: 71.2%
     Bike   : 57.6% (but mean IoU only 0.247 — small parts)
     Health : 46.3%
     Cooking: 44.7%

  8. NEXT STEPS:
     a) Hybrid: VLM bbox → crop → re-prompt SAM3 on cropped region
     b) Error analysis: what does VLM say when it fails?
     c) Full run on 1910 pairs (current coverage: ~75%)
     d) Fix cloud num_predict: 64→128 (25% missing VLM outputs)
""")